# 08 — Gate 2a: θ-tail features + scenarios_v2 (spec v0.10 §2.5 addendum, D-B)

Builds the two **masked-density tail features** (`m_soc_tail`, `biomass_tail`: parent density
where ≥ θ×regional mean, else 0 — so capture = share of **tail mass** and t=1.0 secures the
whole tail), re-audits them under the **unchanged frozen §2.5 rules** (expected:
rare-attainable), renders addendum cards, freezes **`spec/scenarios_v2.json`** (S0–S3
unchanged + tails at t=0; S4 adds the places locks at t=1.0), and refreshes the manifest
(now 10 continuous features, tails ahead of the EFG block — a positional requirement of the
R engine's weight vector). Motivated by M6.7: a total-capture target secures an *amount*,
not *places*. Zero solves. Kernel `y2y-geo`.

In [ ]:
# ---- bootstrap -----------------------------------------------------------------------------
import hashlib, importlib, json, pathlib, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from rasterio.io import MemoryFile
import rasterio.shutil as rio_shutil

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook -- run from within the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)

SPEC = ROOT / "analyses" / "y2y" / "spec"
AUDIT_OBJ = ROOT / "analyses" / "y2y" / "audit" / "audit_objects"
CARDS = ROOT / "analyses" / "y2y" / "audit" / "feature_cards"
T2 = pd.read_csv(AUDIT_OBJ / "feature_characterization.csv").set_index("feature")
pu = lc.pu_mask()
theta = config.AUDIT["theta"]
print(f"PU {int(pu.sum()):,} | theta {theta}x | tails: {dict(config.TAIL_FEATURES)}")

In [ ]:
# ---- build the tail COGs (top level of the stack; masked DENSITY, PU-masked) ---------------
with rasterio.open(config.HANDOFF_DIR / "cost_uniform.tif") as _s:
    PROF = dict(driver="GTiff", width=_s.width, height=_s.height, count=1, dtype="float32",
                crs=_s.crs, transform=_s.transform, nodata=_s.nodata)

def write_cog(path, arr):
    with MemoryFile() as mem:
        with mem.open(**PROF) as tmp:
            tmp.write(np.asarray(arr).astype("float32"), 1)
            rio_shutil.copy(tmp, path, driver="COG", compress="DEFLATE",
                            overview_resampling="average", BIGTIFF="IF_SAFER")

tails = {}
for tail, parent in config.TAIL_FEATURES.items():
    v = lc._read(config.HANDOFF_DIR / f"{parent}.tif")
    cut = theta * float(np.nanmean(v[pu]))
    a = np.where(pu, np.where(np.nan_to_num(v, nan=-1.0) >= cut, v, 0.0), np.nan).astype("float32")
    t_area = float((a[pu] > 0).sum() / pu.sum())
    t_mass = float(np.nansum(a) / np.nansum(v[pu]))
    # the tail must reproduce the FROZEN T2 crossing (area) and archive capture (mass)
    assert abs(t_area - float(T2.loc[parent, "theta_area"])) < 2e-3, f"{tail}: area drifted vs T2"
    assert abs(t_mass - float(T2.loc[parent, "theta_target"])) < 5e-3, f"{tail}: mass drifted vs T2"
    out = config.HANDOFF_DIR / f"{tail}.tif"
    write_cog(out, a)
    tails[tail] = dict(cutoff_t_ha=round(cut, 1), area_pct=round(100 * t_area, 2),
                       mass_share_of_parent=round(t_mass, 4),
                       sha256=hashlib.sha256(out.read_bytes()).hexdigest())
    print(f"{tail}: cutoff {cut:.1f} t/ha | {100*t_area:.2f}% of PU | "
          f"{100*t_mass:.1f}% of {parent} mass -> {out.name}")

In [ ]:
# ---- re-audit under the UNCHANGED frozen rules (expected: rare-attainable) -----------------
rows = []
for tail in config.TAIL_FEATURES:
    v = lc._read(config.HANDOFF_DIR / f"{tail}.tif")[pu]
    cls, lever, target, diag = lc.classify_values(v)
    rows.append(dict(feature=tail, cls=cls, lever=lever, target=target, **diag))
    print(f"{tail}: {cls} (lever: {lever}, target {target}) | {diag}")
    assert cls == "rare-attainable", (
        f"{tail} classified {cls}, not rare-attainable -- the spec's stated expectation "
        "failed; STOP and report before Gate 2b")
ADD = pd.DataFrame(rows).set_index("feature")
ADD["derived_from"] = pd.Series(dict(config.TAIL_FEATURES))
ADD["provenance"] = "post-execution addendum (v0.10 D-B, motivated by M6.7); frozen v0.3 constants"
ADD.to_csv(AUDIT_OBJ / "tail_addendum.csv")
print(f"\nwrote {(_p := AUDIT_OBJ / 'tail_addendum.csv').relative_to(ROOT)} "
      "(the frozen T2 + audit_constants.json are untouched)")

In [ ]:
# ---- addendum cards (bespoke one-pagers; the frozen card machinery is DATASETS-bound) ------
CARDS.mkdir(parents=True, exist_ok=True)
for tail, parent in config.TAIL_FEATURES.items():
    v = lc._read(config.HANDOFF_DIR / f"{tail}.tif")[pu]
    pos = v[v > 0]
    area, capt = lc.lorenz(v)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    axes[0].hist(pos, bins=60, color="#4a6fa5")
    axes[0].set_title(f"positive values (n={pos.size:,} = {100*pos.size/pu.sum():.2f}% of PU)")
    axes[0].set_xlabel("t/ha")
    axes[1].plot(area, capt, color="#4a6fa5")
    axes[1].plot([0, 1], [0, 1], ls=":", c="grey")
    axes[1].set_title("Lorenz (density-descending)")
    axes[1].set_xlabel("area fraction"); axes[1].set_ylabel("captured fraction")
    r = ADD.loc[tail]
    axes[2].axis("off")
    axes[2].text(0.02, 0.95, "\n".join([
        f"{tail}",
        f"derived θ-tail mask of {parent}",
        f"class: {r.cls}  (lever: {r.lever})",
        f"leverage {r.leverage}  cap [{r.cap_min}, {r.cap_max}]",
        "audited under FROZEN v0.3 constants",
        "post-execution addendum (v0.10 D-B; M6.7)",
        "t=0 in every cell (absent); t=1.0 in carbon-forward",
    ]), va="top", family="monospace", fontsize=9)
    fig.suptitle(f"Feature card addendum -- {tail}", y=1.02)
    fig.savefig(CARDS / f"{tail}_addendum.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"card -> audit/feature_cards/{tail}_addendum.pdf")

In [ ]:
# ---- spec/scenarios_v2.json: v1 weights unchanged; tail targets added ----------------------
v1 = json.loads((SPEC / "scenarios_v1.json").read_text())
TAILS0 = {t: 0.0 for t in config.TAIL_FEATURES}
v2 = {"_meta": {
    **v1["_meta"],
    "spec_version": "v0.10",
    "estimator": "mga_maxham_v1",
    "derived_utc_v2": datetime.now(timezone.utc).isoformat(),
    "lineage": "scenarios_v1.json (weights unchanged; tail targets added per v0.10 D-B)",
    "tail_features": tails,
    "tail_fallback": "S4 tail t=0.95 pre-authorized if t=1.0 infeasible/pathological",
}}
for name, sc in v1.items():
    if name == "_meta":
        continue
    sc = dict(sc)
    on = 1.0 if name == "S4_carbon" else 0.0
    sc["targets"] = {**sc["targets"], **({t: on for t in config.TAIL_FEATURES})}
    v2[name] = sc
out = SPEC / "scenarios_v2.json"
out.write_text(json.dumps(v2, indent=2))
print(f"wrote {out.relative_to(ROOT)}")
for name in [k for k in v2 if k != "_meta"]:
    print(f"  {name:<16} targets: {v2[name]['targets']}")

In [ ]:
# ---- refresh the manifest + verify the wiring (10 continuous, tails BEFORE the EFGs) -------
importlib.reload(config)
mp = config.write_manifest()
m = json.loads(pathlib.Path(mp).read_text())
roles = [L["role"] for L in m["layers"]]
names = [L["name"] for L in m["layers"]]
from collections import Counter
print(Counter(roles))
assert Counter(roles)["feature_continuous"] == 10
i_efg = roles.index("feature_efg")
assert all(t in names[:i_efg] for t in config.TAIL_FEATURES), \
    "tails must precede the EFG block (pr_weights is positional)"
assert all(m["params"]["targets"].get(t) == 0.0 for t in config.TAIL_FEATURES), \
    "tail targets must default to 0.0 in the manifest"
print("manifest OK: tails ahead of EFGs, targets 0.0 by default "
      f"(targets = {m['params']['targets']})")

## Next

`09_gate2b_mga.ipynb` (kernel `R (y2y)`): the MGA reference run — anchor (with the t=0
equivalence assert against iter9's 5.362813) + k=50 sweeps at g = 5% / 2% / 10%
(~2.5–3 h total, resumable per g, live internet). Then `10_gate2b_analysis.ipynb`.